In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip install -q transformers accelerate qwen_vl_utils bitsandbytes streamlit pillow

In [ ]:
!pip install -q openpyxl

In [1]:
%%writefile app.py
import streamlit as st
import torch
import os
import zipfile
import shutil
import pandas as pd
import time
from tqdm import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

st.set_page_config(layout="wide", page_title="GeoData Excel Extractor")
st.title("\U0001f5fa\ufe0f GeoTag Data to Excel Extractor (V3)")

if 'session_id' not in st.session_state:
    st.session_state['session_id'] = str(int(time.time()))

SESSION_UPLOAD = f"uploads_{st.session_state['session_id']}"
if not os.path.exists(SESSION_UPLOAD): os.makedirs(SESSION_UPLOAD)

@st.cache_resource
def load_qwen():
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", quantization_config=quant_config, device_map="auto")
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
    return model, processor

Overwriting app.py


In [ ]:
!pip install pyngrok

from pyngrok import ngrok
# Replace with your ngrok token from ngrok.com
ngrok.set_auth_token("Your_Ngrok_Token_Here")

public_url = ngrok.connect(8501)
print(f"Click here to open your App: {public_url}")

In [2]:
import os
import subprocess
import threading
import time
import socket

!fuser -k 8501/tcp
!pkill lt
!pkill ngrok

def run_app():
    with open("streamlit_output.log", "w") as f:
        subprocess.run(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "127.0.0.1", "--server.headless", "true"], stdout=f, stderr=f)

thread = threading.Thread(target=run_app, daemon=True)
thread.start()

print("\u23f3 Loading Qwen2.5-VL...")
for i in range(60):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    result = sock.connect_ex(('127.0.0.1', 8501))
    if result == 0:
        print("\n\u2705 Streamlit is UP!")
        sock.close()
        break
    print(".", end="", flush=True)
    time.sleep(5)
    sock.close()

from pyngrok import ngrok
public_url = ngrok.connect("127.0.0.1:8501", bind_tls=True)
print(f"\n\U0001f517 Access your app here: {public_url}")

⏳ Loading Qwen2.5-VL...
.
✅ Streamlit is UP!
